# Section C — Prediction, pilot sensitivity, and conclusion

The first two code cells are unchanged copies of the source notebook's upload and Section A data-quality/tag setup cells. Run them first to create df and tags in this notebook's own runtime. Their saved outputs are preserved. Section B does not need to be executed to run this notebook. For grading, skip past this prerequisite block to the Section C heading. All Section C code, explanations, saved outputs, and the final conclusion are unchanged.

In [90]:
import os
import pandas as pd

FILE = "stories_transactions.csv"

# Upload the CSV if it isn't in Colab's file panel yet
if not os.path.exists(FILE):
    from google.colab import files
    uploaded = files.upload()                  # pick your CSV
    os.rename(next(iter(uploaded)), FILE)      # keep the name the rest of the code expects

# Load the data
df = pd.read_csv(FILE)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")

#Customer ID dropped after the randomness test
df = df.drop(columns=["Customer ID"])
print(f"After dropping Customer ID: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Loaded 51,177 rows x 14 columns
After dropping Customer ID: 51,177 rows x 13 columns


,Transaction ID,Date,Time,Branch,Order Type,Product,Category,Quantity,Unit Price (USD),Discount Rate,Unit Cost (USD),Line Revenue (USD),Line Gross Profit (USD)
0,SYN-0000001,2026-01-01,22:37:35,Harbour,Takeaway,Fresh Orange Juice,Cold Drinks,1,6.04,0.0,2.12,6.04,3.92
1,SYN-0000002,2026-01-01,17:34:07,Garden,Dine-in,Egg Sandwich,Breakfast,1,8.15,0.0,4.45,8.15,3.70
2,SYN-0000003,2026-01-01,22:45:29,Harbour,Dine-in,Brownie,Desserts,1,5.41,0.0,2.37,5.41,3.04
3,SYN-0000004,2026-01-01,21:36:38,Uptown,Dine-in,Iced Mocha,Cold Coffee,1,6.00,0.0,2.14,6.00,3.86
4,SYN-0000005,2026-01-01,12:34:01,Harbour,Dine-in,Classic Matcha,Matcha & Tea,1,6.77,0.0,2.27,6.77,4.50


In [99]:
# ===== Section A: data quality checks =====
import numpy as np, pandas as pd
raw = pd.read_csv("stories_transactions.csv")          # reload WITH Customer ID for the test below

# 1) Duplicates: count, then drop (assumption: identical lines are double entries)
print("Exact duplicate lines:", raw.duplicated().sum())
raw = raw.drop_duplicates().reset_index(drop=True)
print("Rows after dropping duplicates:", len(raw))

# 2) Missing values: only Customer ID has any
print(raw.isna().sum()[raw.isna().sum() > 0])

# 3) Arithmetic: revenue and gross profit follow the formulas on every row
rev = (raw["Quantity"] * raw["Unit Price (USD)"] * (1 - raw["Discount Rate"])).round(2)
gp  = (raw["Line Revenue (USD)"] - raw["Quantity"] * raw["Unit Cost (USD)"]).round(2)
print("Revenue mismatches:", (abs(rev - raw["Line Revenue (USD)"]) > 0.011).sum(),
      "| Gross profit mismatches:", (abs(gp - raw["Line Gross Profit (USD)"]) > 0.011).sum())

# 4) One fixed price and cost per product per branch, all six months
n = raw.groupby(["Product", "Branch"])[["Unit Price (USD)", "Unit Cost (USD)"]].nunique()
print("Product-branch pairs with more than one price or cost:", int((n > 1).any(axis=1).sum()), "of", len(n))
cost = raw.groupby(["Product", "Branch"])["Unit Cost (USD)"].first().unstack()
ratio = cost["Garden"] / cost["Harbour"]
print(f"Garden cost / Harbour cost: min {ratio.min():.3f}, max {ratio.max():.3f} (every product)")
m = raw.groupby("Branch")[["Line Gross Profit (USD)", "Line Revenue (USD)"]].sum()
print("Gross margin by branch:", (m.iloc[:, 0] / m.iloc[:, 1]).round(3).to_dict())

# 5) Customer IDs: does a customer behave consistently? Compare with randomly shuffled IDs
rng = np.random.default_rng(0)
def top_share(d, col, min_n):
    s = d.groupby("Customer ID")[col].agg(lambda x: x.value_counts().iloc[0] / len(x))
    k = d.groupby("Customer ID").size()
    return s[k >= min_n].mean()
tx = raw.drop_duplicates("Transaction ID").dropna(subset=["Customer ID"])
ln = raw.dropna(subset=["Customer ID"])
for name, d, col, k in [("favorite branch", tx, "Branch", 5), ("favorite product", ln, "Product", 8)]:
    real = top_share(d, col, k)
    fake = np.mean([top_share(d.assign(**{col: rng.permutation(d[col].values)}), col, k) for _ in range(10)])
    print(f"Share of visits to {name}: real customers {real:.3f} vs shuffled IDs {fake:.3f}")
print(
    "Customer-ID checks: the observed average favourite-branch and "
    "favourite-product shares are similar to the averages from ten "
    "shuffled comparisons. These limited checks do not prove that IDs "
    "are random or unusable. We exclude Customer ID from the working "
    "analysis pending validation of its meaning and reliability."
)
df = raw.drop(columns=["Customer ID"])

# ===== Product characteristics (assumption: our own tagging) =====
T = {  # product: (temperature, food_or_drink, milk, sweet, caffeine, perishable)
 "Almond Croissant":("ambient","food",0,1,0,1), "Banana Bread":("ambient","food",0,1,0,1),
 "Blueberry Muffin":("ambient","food",0,1,0,1), "Butter Croissant":("ambient","food",0,0,0,1),
 "Chocolate Cookie":("ambient","food",0,1,0,1), "Chocolate Croissant":("ambient","food",0,1,0,1),
 "Cinnamon Roll":("ambient","food",0,1,0,1), "Pistachio Cookie":("ambient","food",0,1,0,1),
 "Avocado Toast":("hot","food",0,0,0,1), "Egg Sandwich":("hot","food",0,0,0,1),
 "Granola Yoghurt":("cold","food",1,1,0,1), "Coffee Frappe":("cold","drink",1,1,1,0),
 "Cold Brew":("cold","drink",0,0,1,0), "Iced Americano":("cold","drink",0,0,1,0),
 "Iced Latte":("cold","drink",1,0,1,0), "Iced Mocha":("cold","drink",1,1,1,0),
 "Fresh Orange Juice":("cold","drink",0,1,0,1), "Lemonade":("cold","drink",0,1,0,0),
 "Peach Iced Tea":("cold","drink",0,1,1,0), "Sparkling Water":("cold","drink",0,0,0,0),
 "Still Water":("cold","drink",0,0,0,0), "Brownie":("ambient","food",0,1,0,1),
 "Cheesecake Slice":("cold","food",1,1,0,1), "Frozen Yoghurt Cup":("cold","food",1,1,0,1),
 "Tiramisu Cup":("cold","food",1,1,1,1), "Americano":("hot","drink",0,0,1,0),
 "Cappuccino":("hot","drink",1,0,1,0), "Caramel Latte":("hot","drink",1,1,1,0),
 "Espresso":("hot","drink",0,0,1,0), "Flat White":("hot","drink",1,0,1,0),
 "Spanish Latte":("hot","drink",1,1,1,0), "Vanilla Latte":("hot","drink",1,1,1,0),
 "Chai Latte":("hot","drink",1,1,1,0), "Classic Matcha":("hot","drink",1,0,1,0),
 "English Breakfast Tea":("hot","drink",0,0,1,0), "Iced Matcha":("cold","drink",1,0,1,0),
 "Mint Tea":("hot","drink",0,0,0,0), "Strawberry Matcha":("cold","drink",1,1,1,0),
 "Chicken Salad Bowl":("cold","food",0,0,0,1), "Quinoa Bowl":("cold","food",0,0,0,1),
 "Soup of the Day":("hot","food",0,0,0,1), "Chicken Pesto Sandwich":("hot","food",0,0,0,1),
 "Halloumi Wrap":("hot","food",0,0,0,1), "Tuna Sandwich":("cold","food",0,0,0,1),
 "Turkey & Cheese Sandwich":("cold","food",0,0,0,1),
}
tags = pd.DataFrame.from_dict(T, orient="index", columns=["temperature","type","milk","sweet","caffeine","perishable"])
tags.index.name = "Product"
price = df.groupby("Product")["Unit Price (USD)"].mean()
tags["price_tier"] = pd.qcut(price, 3, labels=["low","mid","high"])
tags["category"] = df.groupby("Product")["Category"].first()
assert set(tags.index) == set(df["Product"].unique())
tags.to_csv("product_tags.csv")
print(tags.head())

Exact duplicate lines: 132
Rows after dropping duplicates: 51045
Customer ID    18646
dtype: int64
Revenue mismatches: 0 | Gross profit mismatches: 0
Product-branch pairs with more than one price or cost: 0 of 135
Garden cost / Harbour cost: min 1.426, max 1.439 (every product)
Gross margin by branch: {'Garden': 0.469, 'Harbour': 0.65, 'Uptown': 0.626}
Share of visits to favorite branch: real customers 0.571 vs shuffled IDs 0.568
Share of visits to favorite product: real customers 0.165 vs shuffled IDs 0.170
Customer-ID checks: the observed average favourite-branch and favourite-product shares are similar to the averages from ten shuffled comparisons. These limited checks do not prove that IDs are random or unusable. We exclude Customer ID from the working analysis pending validation of its meaning and reliability.
                 temperature  type  milk  sweet  caffeine  perishable  \
Product                                                                 
Almond Croissant     ambien

## **Section C**

## How sure are we, and how can Stories test it?

We compare product-sales predictions with simple baselines by hiding one product at a time and predicting its sales using the remaining products.

We then estimate how large a change a branch pilot might detect, outline practical tests, and separate measured findings from assumptions and missing information.

In [109]:
# SECTION C1 — Can product characteristics predict sales?
# Leave-one-product-out evaluation of chain-wide product sales.
# Does not modify df, tags, or prod_idx.

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold

# 1. Validate and align products.
c1_required_tags = [
    "temperature", "type", "milk", "sweet",
    "caffeine", "perishable", "category"
]

assert tags.index.is_unique, "Duplicate product names in tags."
assert set(c1_required_tags).issubset(tags.columns), "Missing feature columns."
assert df[
    ["Product", "Branch", "Quantity", "Unit Price (USD)"]
].notna().all().all(), "Missing required transaction values."
assert df["Quantity"].gt(0).all(), "Review non-positive quantities."
assert df["Unit Price (USD)"].gt(0).all(), "Prices must be positive."

c1_units = df.groupby("Product")["Quantity"].sum().sort_index()

assert set(c1_units.index) == set(tags.index), (
    "Transaction products and tagged products do not match."
)
assert len(c1_units) >= 7, "Too few products for this evaluation."

c1_features = tags.loc[c1_units.index, c1_required_tags].copy()
assert c1_features.notna().all().all(), "Missing product features."

for col in ["milk", "sweet", "caffeine", "perishable"]:
    assert c1_features[col].isin([0, 1]).all(), (
        f"{col} must contain only 0 and 1."
    )

# A proposed item's price is known before launch.
# Use equal-weight branch list prices rather than weighting by sales.
c1_price_counts = df.groupby(
    ["Product", "Branch"]
)["Unit Price (USD)"].nunique()

assert c1_price_counts.eq(1).all(), (
    "Some product/branch list prices vary. Review price construction first."
)

c1_prices = (
    df.groupby(["Product", "Branch"])["Unit Price (USD)"]
    .first()
    .groupby("Product")
    .mean()
    .reindex(c1_units.index)
)

c1_features["log_price"] = np.log(c1_prices)
c1_log_units = np.log(c1_units)

c1_numeric = [
    "milk", "sweet", "caffeine", "perishable", "log_price"
]

# 2. Create fresh preprocessing for each fitted model.
def c1_pipeline(include_category, estimator):
    categorical = ["temperature", "type"]
    if include_category:
        categorical += ["category"]

    preprocessing = ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                categorical
            ),
            ("numeric", "passthrough", c1_numeric)
        ],
        remainder="drop"
    )

    return Pipeline([
        ("features", preprocessing),
        ("scale", StandardScaler()),
        ("model", estimator)
    ])


# 3. Hold out each product and predict from all other products.
c1_prediction_rows = []
c1_ridge_alphas = []
c1_category_fallbacks = 0

for position, product in enumerate(c1_units.index, start=1):
    train_mask = c1_units.index != product

    X_train = c1_features.loc[train_mask]
    X_test = c1_features.loc[[product]]
    y_train = c1_log_units.loc[train_mask]

    # Log-scale means correspond to geometric means of unit sales.
    overall_prediction = y_train.mean()

    category = c1_features.loc[product, "category"]
    same_category = X_train["category"].eq(category)

    if same_category.any():
        category_prediction = y_train.loc[same_category].mean()
    else:
        category_prediction = overall_prediction
        c1_category_fallbacks += 1

    knn = c1_pipeline(
        include_category=False,
        estimator=KNeighborsRegressor(n_neighbors=5)
    )
    knn.fit(X_train, y_train)
    knn_prediction = knn.predict(X_test)[0]

    # Inner CV selects alpha without seeing the held-out product.
    # Encoding and scaling are refitted within every inner training fold.
    ridge_search = GridSearchCV(
        estimator=c1_pipeline(
            include_category=True,
            estimator=Ridge()
        ),
        param_grid={
            "model__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000]
        },
        scoring="neg_mean_absolute_error",
        cv=KFold(n_splits=5, shuffle=True, random_state=42),
        n_jobs=1,
        error_score="raise"
    )
    ridge_search.fit(X_train, y_train)
    ridge_prediction = ridge_search.predict(X_test)[0]

    c1_ridge_alphas.append({
        "Product": product,
        "selected_alpha": ridge_search.best_params_["model__alpha"]
    })

    predictions = {
        "Overall geometric mean": overall_prediction,
        "Category geometric mean (baseline)": category_prediction,
        "5-neighbour kNN: characteristics": knn_prediction,
        "Ridge: characteristics + category": ridge_prediction
    }

    for method, predicted_log in predictions.items():
        actual = float(c1_units.loc[product])
        predicted = float(np.exp(predicted_log))

        c1_prediction_rows.append({
            "Product": product,
            "Category": category,
            "Method": method,
            "actual_units": actual,
            "predicted_units": predicted,
            "absolute_error_units": abs(predicted - actual),
            "absolute_percentage_error": (
                100 * abs(predicted - actual) / actual
            ),
            "absolute_log_error": abs(
                predicted_log - np.log(actual)
            )
        })

    if position % 10 == 0 or position == len(c1_units):
        print(f"Evaluated {position}/{len(c1_units)} products.")

c1_predictions = pd.DataFrame(c1_prediction_rows)
c1_selected_alphas = pd.DataFrame(c1_ridge_alphas)

# 4. Summarize held-out prediction errors.
c1_results = c1_predictions.groupby("Method", sort=False).agg(
    products_tested=("Product", "size"),
    mean_absolute_error_units=("absolute_error_units", "mean"),
    MAPE_pct=("absolute_percentage_error", "mean"),
    mean_absolute_log_error=("absolute_log_error", "mean")
)

c1_results["multiplicative_error_factor"] = np.exp(
    c1_results["mean_absolute_log_error"]
)
c1_results["log_error_equivalent_pct"] = (
    100 * (c1_results["multiplicative_error_factor"] - 1)
)

print("\nMODEL COMPARISON — lower error is better")
display(c1_results.round(3))

print(f"\nCategory baseline fallbacks: {c1_category_fallbacks}")

# Directly compare ridge with the category baseline on the same products.
c1_error_comparison = c1_predictions.pivot(
    index="Product",
    columns="Method",
    values="absolute_log_error"
)

c1_baseline_name = "Category geometric mean (baseline)"
c1_ridge_name = "Ridge: characteristics + category"

c1_baseline_error = c1_error_comparison[c1_baseline_name].mean()
c1_ridge_error = c1_error_comparison[c1_ridge_name].mean()

print(
    "Products where ridge has lower absolute log error than the baseline:",
    int(
        (
            c1_error_comparison[c1_ridge_name]
            < c1_error_comparison[c1_baseline_name]
        ).sum()
    ),
    "of",
    len(c1_units)
)

if c1_baseline_error > 0:
    print(
        "Ridge reduction in mean absolute log error versus baseline:",
        f"{100 * (1 - c1_ridge_error / c1_baseline_error):.2f}%"
    )

print("\nTEN LARGEST RIDGE ERRORS")
display(
    c1_predictions[
        c1_predictions["Method"].eq(c1_ridge_name)
    ]
    .sort_values("absolute_log_error", ascending=False)
    .head(10)[[
        "Product", "Category", "actual_units",
        "predicted_units", "absolute_percentage_error"
    ]]
    .round(2)
)

print("""
HOW TO READ THE RESULTS
• MAPE is mean absolute percentage error across held-out products.
• Log-error equivalent is a different metric, included for comparison
  with the brief. A value of 25% means a multiplicative error factor
  of 1.25; it is not a confidence interval or a promise of ±25% accuracy.
• Each product receives equal weight in these error summaries.
• All methods use the same held-out products.
• The model predicts chain-wide sales over the observed period.
  It does not identify the best branch for a new product.
• Held-out existing products are only a proxy for new launches.
  Launch effects, availability, and cannibalization are not measured.
• Tags must describe product characteristics rather than be derived
  from sales or popularity.
• We have not assumed that ridge will outperform the category baseline.
""")

Evaluated 10/45 products.
Evaluated 20/45 products.
Evaluated 30/45 products.
Evaluated 40/45 products.
Evaluated 45/45 products.

MODEL COMPARISON — lower error is better


,products_tested,mean_absolute_error_units,MAPE_pct,mean_absolute_log_error,multiplicative_error_factor,log_error_equivalent_pct
Method,,,,,,
Overall geometric mean,45,338.331,32.855,0.297,1.346,34.622
Category geometric mean (baseline),45,308.957,28.022,0.258,1.294,29.421
5-neighbour kNN: characteristics,45,296.966,28.307,0.263,1.300,30.032
Ridge: characteristics + category,45,248.484,23.638,0.223,1.249,24.921



Category baseline fallbacks: 0
Products where ridge has lower absolute log error than the baseline: 24 of 45
Ridge reduction in mean absolute log error versus baseline: 13.72%

TEN LARGEST RIDGE ERRORS


,Product,Category,actual_units,predicted_units,absolute_percentage_error
83,English Breakfast Tea,Matcha & Tea,473.0,1082.15,128.79
71,Coffee Frappe,Cold Coffee,956.0,1588.56,66.17
103,Granola Yoghurt,Breakfast,575.0,925.29,60.92
55,Chocolate Cookie,Bakery,1662.0,1053.43,36.62
111,Iced Americano,Cold Coffee,1542.0,987.90,35.93
167,Tiramisu Cup,Desserts,1032.0,1521.81,47.46
19,Blueberry Muffin,Bakery,928.0,1348.12,45.27
163,Strawberry Matcha,Matcha & Tea,1481.0,1026.80,30.67
39,Chai Latte,Matcha & Tea,1122.0,798.04,28.87
91,Flat White,Hot Coffee,1370.0,1924.04,40.44



HOW TO READ THE RESULTS
• MAPE is mean absolute percentage error across held-out products.
• Log-error equivalent is a different metric, included for comparison
  with the brief. A value of 25% means a multiplicative error factor
  of 1.25; it is not a confidence interval or a promise of ±25% accuracy.
• Each product receives equal weight in these error summaries.
• All methods use the same held-out products.
• The model predicts chain-wide sales over the observed period.
  It does not identify the best branch for a new product.
• Held-out existing products are only a proxy for new launches.
  Launch effects, availability, and cannibalization are not measured.
• Tags must describe product characteristics rather than be derived
  from sales or popularity.
• We have not assumed that ridge will outperform the category baseline.



C1 — What the prediction results show

We tested all 45 products by hiding one product at a time and predicting its total sales from the other 44. Feature preprocessing and ridge tuning used only training products.

Using the log-error-equivalent measure, lower is better:

Overall geometric mean: 34.6%.
Category geometric mean: 29.4%.
Five-neighbour model using characteristics: 30.0%.
Ridge regression using characteristics and category: 24.9%.

The characteristics-only neighbour model did not outperform the category baseline on this measure. Ridge performed better overall, reducing mean absolute log error by 13.7% compared with the category baseline.

However, ridge improved predictions for only 24 of 45 products. Some individual errors remained large: it predicted approximately 1,082 English Breakfast Tea units when actual sales were 473.

Ridge’s mean absolute percentage error was 23.6%, compared with 28.0% for the category baseline. This differs from the log-error-equivalent measure above. Neither measure is a confidence interval or guarantees that an individual prediction will fall within those percentages.

The results support using ridge as a preliminary estimate alongside the simple category baseline. They do not establish that it will reliably outperform the baseline on future launches.

This model predicts total sales across the chain during the observed six-month period. It does not identify the best branch for a new item. Holding out existing products approximates an unseen-product test, but does not capture launch effects, changes in availability, or sales taken away from existing products.

Any new-item forecast should therefore be checked through a small real-world pilot before wider rollout.

In [110]:
# SECTION C2 — How large a change could an Uptown pilot detect?
# Does not modify df, tags, or prod_idx.

import pandas as pd
import numpy as np
from scipy.stats import norm
from IPython.display import display

c2_data = df.copy()
c2_data["Date"] = pd.to_datetime(
    c2_data["Date"], format="%Y-%m-%d", errors="raise"
).dt.normalize()

c2_required = [
    "Date", "Branch", "Product", "Quantity",
    "Unit Cost (USD)", "Line Gross Profit (USD)"
]
assert c2_data[c2_required].notna().all().all(), (
    "Missing required values."
)
assert c2_data["Quantity"].gt(0).all(), (
    "Review non-positive quantities."
)

c2_branches = ["Harbour", "Garden", "Uptown"]
assert set(c2_branches).issubset(c2_data["Branch"].unique())

# 1. Daily branch GP, with explicit coverage checks.
c2_calendar = pd.date_range(
    c2_data["Date"].min(), c2_data["Date"].max()
)

c2_daily_gp = (
    c2_data.groupby(["Date", "Branch"])["Line Gross Profit (USD)"]
    .sum()
    .unstack("Branch")
    .reindex(index=c2_calendar, columns=c2_branches)
)

assert c2_daily_gp.notna().all().all(), (
    "Missing branch-day records. Investigate before estimating variability."
)

# Monday–Sunday weeks; retain only weeks with seven observed days.
c2_weekly_gp = c2_daily_gp.resample("W-SUN").sum()
c2_week_day_counts = c2_daily_gp.resample("W-SUN").size()
c2_weekly_gp = c2_weekly_gp.loc[c2_week_day_counts.eq(7)].copy()

assert len(c2_weekly_gp) >= 12, (
    "Too few complete weeks for this planning calculation."
)
assert c2_weekly_gp.gt(0).all().all(), (
    "Weekly GP must be positive for log-ratio calculations."
)

# Uptown's weekly GP relative to the combined comparison branches.
c2_log_ratio = np.log(
    c2_weekly_gp["Uptown"]
    / (c2_weekly_gp["Harbour"] + c2_weekly_gp["Garden"])
)

c2_sd = c2_log_ratio.std(ddof=1)
c2_lag1 = c2_log_ratio.autocorr(lag=1)

print("1. HISTORICAL VARIABILITY")
print(f"Complete Monday–Sunday weeks: {len(c2_weekly_gp)}")
print(
    "First complete week:",
    (c2_weekly_gp.index.min() - pd.Timedelta(days=6)).date(),
    "to", c2_weekly_gp.index.min().date()
)
print(
    "Last complete week:",
    (c2_weekly_gp.index.max() - pd.Timedelta(days=6)).date(),
    "to", c2_weekly_gp.index.max().date()
)
print(f"Standard deviation of weekly log GP ratios: {c2_sd:.4f}")
print(
    "Approximate relative variability:",
    f"{100 * c2_sd:.2f}%"
)
print(f"Lag-one correlation of weekly log ratios: {c2_lag1:.3f}")

# 2. Approximate minimum detectable effect (MDE).
# Two-sided 5% significance, 80% power.
# Equal numbers of baseline and pilot weeks; independent weekly observations.
c2_z = norm.ppf(0.975) + norm.ppf(0.80)
c2_mde_rows = []

for weeks in [4, 6, 8, 12]:
    mde_log = c2_z * c2_sd * np.sqrt(2 / weeks)

    c2_mde_rows.append({
        "baseline_weeks": weeks,
        "pilot_weeks": weeks,
        "total_weeks": 2 * weeks,
        "MDE_log_units": mde_log,
        "detectable_increase_pct": 100 * np.expm1(mde_log),
        "detectable_decrease_pct": 100 * (1 - np.exp(-mde_log))
    })

c2_mde = pd.DataFrame(c2_mde_rows)

print("\n2. APPROXIMATE PILOT SENSITIVITY")
display(c2_mde.round(3))

# 3. Soup's observed contribution and hypothetical waste threshold.
c2_uptown = c2_data[c2_data["Branch"].eq("Uptown")]
c2_soup = c2_uptown[
    c2_uptown["Product"].eq("Soup of the Day")
]

assert not c2_soup.empty, "Soup of the Day was not found at Uptown."

c2_days = len(c2_calendar)
c2_soup_units = c2_soup["Quantity"].sum()
c2_soup_gp = c2_soup["Line Gross Profit (USD)"].sum()
c2_uptown_gp = c2_uptown["Line Gross Profit (USD)"].sum()

assert c2_uptown_gp > 0
c2_soup_cost = (
    c2_soup["Quantity"] * c2_soup["Unit Cost (USD)"]
).sum() / c2_soup_units
assert c2_soup_cost > 0

c2_soup_summary = pd.Series({
    "observed_calendar_days": c2_days,
    "soup_units_sold": c2_soup_units,
    "soup_units_per_day": c2_soup_units / c2_days,
    "soup_recorded_gp_usd": c2_soup_gp,
    "soup_gp_per_day_usd": c2_soup_gp / c2_days,
    "soup_share_of_Uptown_gp_pct": (
        100 * c2_soup_gp / c2_uptown_gp
    ),
    "weighted_soup_unit_cost_usd": c2_soup_cost,
    "break_even_waste_portions_per_day": (
        (c2_soup_gp / c2_days) / c2_soup_cost
        if c2_soup_gp >= 0 else np.nan
    )
}, name="Value")

print("\n3. SOUP'S SCALE AND WASTE SENSITIVITY")
display(c2_soup_summary.to_frame().round(3))

print("""
INTERPRETATION AND ASSUMPTIONS
• The variability measure describes weekly log GP ratios around their
  historical mean; it is not the standard deviation of weekly changes.
• A six-week pilot here requires six baseline weeks as well.
• Detection estimates assume independent weeks, stable variability,
  comparable branch trends, and no spillovers to comparison branches.
• Serial correlation, seasonality, or different branch trends can make
  these approximations inaccurate. Lag-one correlation is only a diagnostic.
• Historical weeks include the Harbour promotions. The calculation
  describes observed variability, not a clean experimental control period.
• Increase and decrease percentages differ because log effects are
  converted back to ordinary percentages.
• Soup's GP share is a scale comparison, not the predicted effect of
  removing soup. Substitution, lost companion sales, and avoided waste
  could change the result.
• Break-even waste assumes each discarded portion costs the same as a
  sold portion, with waste not already included in recorded unit costs.
• Record actual disposals and opening/closing usable stock in a waste log.
  Prepared minus sold is not necessarily waste.
• These estimates do not establish that soup should be removed.
""")

1. HISTORICAL VARIABILITY
Complete Monday–Sunday weeks: 25
First complete week: 2026-01-05 to 2026-01-11
Last complete week: 2026-06-22 to 2026-06-28
Standard deviation of weekly log GP ratios: 0.0711
Approximate relative variability: 7.11%
Lag-one correlation of weekly log ratios: 0.164

2. APPROXIMATE PILOT SENSITIVITY


,baseline_weeks,pilot_weeks,total_weeks,MDE_log_units,detectable_increase_pct,detectable_decrease_pct
0,4,4,8,0.141,15.125,13.138
1,6,6,12,0.115,12.188,10.864
2,8,8,16,0.100,10.472,9.480
3,12,12,24,0.081,8.472,7.810



3. SOUP'S SCALE AND WASTE SENSITIVITY


,Value
observed_calendar_days,181.000
soup_units_sold,75.000
soup_units_per_day,0.414
soup_recorded_gp_usd,311.440
soup_gp_per_day_usd,1.721
soup_share_of_Uptown_gp_pct,0.895
weighted_soup_unit_cost_usd,2.680
break_even_waste_portions_per_day,0.642



INTERPRETATION AND ASSUMPTIONS
• The variability measure describes weekly log GP ratios around their
  historical mean; it is not the standard deviation of weekly changes.
• A six-week pilot here requires six baseline weeks as well.
• Detection estimates assume independent weeks, stable variability,
  comparable branch trends, and no spillovers to comparison branches.
• Serial correlation, seasonality, or different branch trends can make
  these approximations inaccurate. Lag-one correlation is only a diagnostic.
• Historical weeks include the Harbour promotions. The calculation
  describes observed variability, not a clean experimental control period.
• Increase and decrease percentages differ because log effects are
  converted back to ordinary percentages.
• Soup's GP share is a scale comparison, not the predicted effect of
  removing soup. Substitution, lost companion sales, and avoided waste
  could change the result.
• Break-even waste assumes each discarded portion costs the sa

C2 — What the pilot-sensitivity results show

We analysed 25 complete Monday–Sunday weeks. The standard deviation of Uptown’s weekly log gross-profit ratio relative to Harbour and Garden was 0.0711, approximately 7.1% in relative terms.

Under an approximate calculation using a two-sided 5% significance level and 80% power, six baseline weeks followed by six pilot weeks could detect an increase of approximately 12.2% or a decrease of approximately 10.9% in Uptown’s gross profit relative to the comparison branches.

These are planning estimates, not guaranteed thresholds. The calculation assumes independent weeks, stable variability, and comparable branch trends. The observed lag-one correlation was 0.164, and the historical period includes Harbour’s promotions. These factors limit the calculation’s precision.

Soup of the Day sold 75 units at Uptown over 181 days, averaging 0.414 portions per day. It generated USD 311.44 in recorded gross profit, or approximately USD 1.72 per day, representing 0.895% of Uptown’s recorded gross profit.

Soup’s contribution is much smaller than the estimated detection threshold. A six-week pilot is therefore unlikely to detect a branch-level effect as small as soup’s current contribution. Finding no statistically detectable change would not prove that changing soup preparation had no effect.

The actual effect could differ because customers might switch products, companion purchases might change, or waste could be avoided. Soup’s current gross-profit share is not a forecast of the effect of removing it.

At the observed sales rate and unit cost, approximately 0.642 discarded portions per day would consume soup’s recorded gross profit. This assumes discarded portions cost USD 2.68 each and waste is not already included in recorded costs.

The pilot should therefore measure actual disposals, preparation quantities, opening and closing usable stock, stockouts, and changes in sales. Prepared quantity minus sold quantity is not necessarily waste.

These findings support collecting better evidence before changing the menu. They do not establish that soup should be removed.

C3 — Practical tests before wider changes

1. Waste and availability log

For four weeks, record daily opening usable stock, portions prepared or received, portions sold, transfers, staff meals, actual disposals, and closing usable stock for perishable products at each branch.

Record disposal reasons and any periods when products are unavailable. Prepared minus sold is not automatically waste: unsold portions may remain usable or have another recorded destination.

Value discarded portions using verified avoidable costs. Compare the results with Section B’s break-even waste estimates. For Uptown’s soup, the historical threshold is approximately 0.642 portions per day, assuming the recorded unit cost applies to discarded portions.

Recalculate the threshold if sales or costs change. Exceeding it supports investigating preparation changes, but does not automatically justify removing the product.


2. Uptown soup-preparation pilot

Use six baseline weeks followed by six pilot weeks. Collect the waste and availability log throughout both periods; the initial four-week log can form part of the baseline.

Test one clearly defined change at Uptown, such as a smaller daily batch. Set the batch quantity after reviewing baseline demand and waste, and document the rule before the pilot begins. Keep soup preparation unchanged at Harbour and Garden.

Track waste costs, soup sales, stockouts, customer requests that cannot be fulfilled, preparation time, and total branch gross profit. Record concurrent changes in prices, promotions, opening hours, or operations.

Compare Uptown’s baseline-to-pilot change with both comparison branches separately and together. Account for weekday patterns and check whether the branches had comparable trends before the change.

Judge the pilot using measured waste savings alongside sales and service effects. Subtract waste costs from recorded gross profit only if they are not already included. Account separately for additional avoidable labour or other implementation costs.

Agree on acceptable stockout and service limits before starting. Restore the previous preparation approach if those limits are breached.

A six-week pilot is unlikely to detect a branch-level effect as small as soup’s current contribution. No detectable change in total gross profit must not be interpreted as proof that the pilot had no effect.


3. Garden cost audit

Compare supplier invoices, purchase units and pack sizes, recipe quantities, portion sizes, transfer prices, and cost-allocation rules for the same products across all three branches.

Start with products contributing the largest benchmark cost differences, including Iced Matcha and Chicken Pesto Sandwich, then check whether the same explanation applies across the menu.

Determine whether Garden’s premium represents higher avoidable purchase costs, different product specifications, allocated overhead, or a recording rule.

Section B’s peer-mean difference of USD 16,628.98 is a six-month benchmark, not a savings commitment. Only verified avoidable cost differences should enter a savings estimate.


4. Future discount test

Before repeating a discount, define the products, discount rate, primary financial outcome, test duration, comparison method, and stopping rules.

With only three branches, consider randomizing discount and no-discount periods within branches, balanced by weekday. Allow a gap between periods if promotions may affect later purchases. Check that customers switching branches would not undermine the comparison.

Choose the duration using historical variability and the smallest gross-profit change worth detecting. A longer test alone does not guarantee a reliable result.

Measure total branch gross profit alongside promoted-category gross profit, units sold, waste, and campaign costs. This helps identify whether discounted products replace full-price purchases elsewhere on the menu.

Judge success by the change in financial contribution after relevant costs, not by sales growth alone. Report uncertainty and differences between branches.


5. New-item pilot

Use the ridge forecast and category baseline as preliminary estimates, not sales commitments. The current model predicts chain-wide sales and cannot select the best branch by itself.

Choose a pilot branch using separately verified branch-level category evidence and operational feasibility. Begin with limited preparation and record availability, sales, waste, customer feedback, and effects on existing products.

Compare results with expectations before expanding. Investigate forecast errors rather than assuming that an unsuccessful forecast proves there is no demand.

C4 — What we measured, what is missing, and what we assumed

Measured findings

The dataset covers January–June 2026 across three branches and 45 products.

Using the team’s perishable tags, 13 branch–product combinations sell below one unit per day: 3 at Garden and 10 at Uptown.

Between 60.6% and 67.1% of baskets containing water or espresso also contain another profitable product.

Garden’s recorded costs are approximately 43% above Harbour’s and 49% above Uptown’s when comparing the same products.

Harbour’s promotion results depend on the comparison branch. March is inconclusive, while May shows a consistent estimated gross-profit shortfall. These are comparison-based estimates, not proven causal effects.

In the held-out-product test, ridge achieved a mean absolute percentage error of 23.6%, compared with 28.0% for the category baseline. This measures performance on existing products whose sales were hidden during training, not actual new launches.

Soup contributes approximately 0.895% of Uptown’s recorded gross profit. The pilot-detection estimates are much larger than this contribution and depend on statistical assumptions.


Missing information

Waste and inventory: We do not observe preparation quantities, usable stock, actual disposals, or transfers. Actual waste costs cannot be calculated.

Availability: We cannot distinguish weak demand from low sales caused by stockouts or limited availability.

Operating costs: Rent, labour, marketing, and other operating expenses are unavailable. Recorded gross profit is not net profit.

Customers: Customer IDs exist in the source data, but their meaning and reliability are unconfirmed. Our limited checks found similar favourite-branch and favourite-product shares in observed and shuffled data, so we excluded IDs from the working analysis (Section A). We analyse baskets, but do not analyse repeat-customer behaviour or individual substitution after menu changes.

Cost explanations: Supplier invoices, pack sizes, recipes, transfer prices, and allocation rules are needed to explain Garden’s premium.

Promotion effects: The two observed Harbour promotion windows do not provide randomized evidence. Other branch-specific changes could affect the estimates.

New launches: No actual new-item launch is evaluated by the model. Launch effects and sales taken from existing products are unknown.

Time coverage: Six months cannot establish full-year seasonality or performance after June.


Assumptions and how they affect interpretation

Perishable tags correctly identify products exposed to spoilage. If the tags are wrong, the slow-perishable results change.

Products were available throughout the analysis period. If availability differed, low recorded sales may understate demand.

Transaction IDs identify complete baskets. If items from one purchase are split across IDs, co-purchase rates may be understated.

Recorded unit costs apply to discarded portions and exclude waste already incurred. If this is false, the waste thresholds must be recalculated to avoid incorrect costing or double counting.

Garden’s peer-cost scenarios hold its quantities and revenue fixed. They do not establish that peer costs are achievable or that changing procurement would leave quality and sales unchanged.

The promotion comparison assumes Harbour would have followed similar proportional trends to the comparison branches without the discount. Control sensitivity and placebo differences weaken this assumption.

The product tags and list-price features are available before launch and are not derived from popularity. Otherwise, the model would use information unavailable for a genuinely new product.

Existing products are informative about future launches. A new product outside the observed range may perform very differently.

Pilot sensitivity assumes independent weeks and stable relative variability. Serial correlation, seasonality, and changing branch trends can alter the detection thresholds.

We treat the 132 identical duplicate lines as accidental double entries (Section A). They represent approximately 0.26% of the original rows. If they were legitimate repeated line items, removing them would understate sales; the effect on individual findings has not been tested.


How these uncertainties affect decisions

Low waste would weaken the case for reducing preparation solely because sales are slow. High waste would support testing smaller batches or other preparation changes before deciding on removal.

Customer substitution could preserve profit after a menu change, while lost companion purchases could reduce it. Neither effect is established by the current basket analysis.

If Garden’s premium is allocated overhead rather than avoidable purchase cost, changing the recorded allocation would not itself create cash savings.

A promotion that increases units can still reduce gross profit. A pilot with no statistically detectable change can still have a financially relevant effect that the test was too small to detect.

The evidence supports targeted tests and a cost audit. It does not yet justify claiming that the identical menu causes the financial differences or that every benchmark difference can be recovered as savings.

Final conclusion — What Stories should test next

The branches differ in sales volume, product sales mix, and recorded costs. These differences create financial concerns, but the data does not prove that an identical menu causes them.

Our findings support targeted investigation and small tests rather than immediate menu removals.

First, audit Garden’s costs. Matching products and holding Garden’s quantities fixed gives a six-month cost difference of USD 15,887.50 against Harbour and USD 17,370.46 against Uptown. The peer-mean benchmark is USD 16,628.98. This is the largest quantified benchmark difference, but invoices, recipes, and accounting rules must establish whether any of it is avoidable.

Second, measure waste before changing preparation. Thirteen branch–product combinations sell below one unit per day. Low sales alone do not establish losses, and smaller branch volume explains much of the pattern. Uptown’s soup reaches its hypothetical waste break-even threshold at approximately 0.642 discarded portions per day. Actual disposals and availability should guide a smaller-batch pilot.

Third, evaluate discounts on gross profit rather than units sold. The Harbour promotions required approximately 28% more combined Cold Coffee and Dessert units to preserve gross profit at the original product mix. March’s financial effect is inconclusive. May shows a comparison-based gross-profit shortfall of approximately USD 332 using pooled controls, although placebo differences prevent a firm causal conclusion.

Fourth, do not judge water and espresso solely by their low profit per unit. They contribute positive recorded gross profit, and approximately 61%–67% of their baskets contain another profitable product. This does not establish that they cause companion purchases, but it argues against assuming that low standalone profit means an item has no value.

Finally, use prediction models as preliminary estimates. Ridge’s held-out-product mean absolute percentage error was 23.6%, compared with 28.0% for the category baseline. Individual errors can still be large, and the model does not select a launch branch. New products need small real-world pilots.

The proposed next actions are a Garden cost audit, a waste and availability log, a controlled preparation pilot, and better-designed promotion tests.

The project’s financial figures describe different scenarios and comparisons. They must not be added together as a total loss or promised saving.